# C2-linear-models — Practice p20

**Type:** constrained coding · **Difficulty:** core · **Concepts:** linear-regression-estimator-derivation

Implement \`ols_full_rank(X, y)\`; an intercept is already a column of
$X$.

Accept finite numeric \`X (n, p)\` and \`y (n,)\` only when $n\ge p$,
row counts match, and \`np.linalg.matrix_rank(X) == p\`.
Raise \`ValueError\` before solving for malformed, non-finite, or
rank-deficient input. Do not mutate inputs.

Form $G=X^TX$ and $c=X^Ty$, then make exactly one
\`np.linalg.solve(G, c)\` call. If floating-point formation makes $G$
numerically singular and that solve raises \`np.linalg.LinAlgError\`,
convert it to \`ValueError\` after the one attempted call. This is the
normal-equation method's numerical limitation; grading success fixtures
have a solvable computed Gram matrix.

On success, return finite float \`beta (p,)\` satisfying the coefficient,
residual, normal-system, and orthogonality contracts with
\`ATOL = 1e-10\`, \`RTOL = 1e-10\`. Relative tolerance is required
because equivalent well-conditioned problems may be rescaled; zero
orthogonality uses a norm bound
\`ATOL + RTOL * problem_scale\`, not ineffective relative tolerance
against literal zero.

**Banned inside the function and any helper it calls (zero points):**
\`np.linalg.inv\`, \`np.linalg.pinv\`, \`np.linalg.lstsq\`,
\`getattr\`, \`__dict__\`, any spelling of \`sklearn\` or
\`statsmodels\`, loops, comprehensions, recursion, or saved aliases to
forbidden routines.

The immutable checker instruments required/forbidden calls, inspects
nested code and referenced globals/defaults/closures, uses scaled
secondary fixtures, and verifies that output propagates from the solve
return rather than a dummy call.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 1e-10


def ols_full_rank(X, y):
    # YOUR CODE HERE
    ...

## Immutable contract check — do not edit

The checker independently verifies accepted, scaled, rejected, and
computed-Gram-singular cases. It checks exact solve arguments, patches
every forbidden NumPy route, recursively audits code and captured
aliases, and uses a controlled solve return to prove data flow.

In [ ]:
import dis
import functools
import inspect
import types

_ORIGINAL_SOLVE_P20 = np.linalg.solve
_ORIGINAL_INV_P20 = np.linalg.inv
_ORIGINAL_PINV_P20 = np.linalg.pinv
_ORIGINAL_LSTSQ_P20 = np.linalg.lstsq
_FORBIDDEN_FUNCS_P20 = (
    _ORIGINAL_INV_P20, _ORIGINAL_PINV_P20, _ORIGINAL_LSTSQ_P20,
)
_FLOAT_EPS_P20 = np.finfo(float).eps
_BACKWARD_SAFETY_P20 = 64.0


def _audit_value_p20(value, seen, pending_functions):
    marker = id(value)
    if marker in seen:
        return
    seen.add(marker)
    assert all(value is not item for item in _FORBIDDEN_FUNCS_P20)

    if isinstance(value, functools.partial):
        _audit_value_p20(value.func, seen, pending_functions)
        _audit_value_p20(value.args, seen, pending_functions)
        _audit_value_p20(value.keywords or {}, seen, pending_functions)
    elif isinstance(value, types.FunctionType):
        pending_functions.append(value)
    elif isinstance(value, types.MethodType):
        _audit_value_p20(value.__func__, seen, pending_functions)
        _audit_value_p20(value.__self__, seen, pending_functions)
    elif isinstance(value, dict):
        for key, item in value.items():
            _audit_value_p20(key, seen, pending_functions)
            _audit_value_p20(item, seen, pending_functions)
    elif isinstance(value, (tuple, list, set, frozenset)):
        for item in value:
            _audit_value_p20(item, seen, pending_functions)
    elif isinstance(value, (types.ModuleType, type)):
        return
    else:
        try:
            attributes = vars(value)
        except TypeError:
            attributes = None
        if attributes is not None:
            _audit_value_p20(attributes, seen, pending_functions)
        if callable(value):
            call_impl = type(value).__call__
            if isinstance(call_impl, types.FunctionType):
                pending_functions.append(call_impl)


_pending_functions_p20 = [ols_full_rank]
_seen_functions_p20 = set()
_codes_p20 = []
while _pending_functions_p20:
    _function_p20 = _pending_functions_p20.pop()
    if id(_function_p20) in _seen_functions_p20:
        continue
    _seen_functions_p20.add(id(_function_p20))
    _defaults_p20 = (
        tuple(_function_p20.__defaults__ or ())
        + tuple((_function_p20.__kwdefaults__ or {}).values())
    )
    for _value_p20 in _defaults_p20:
        _audit_value_p20(_value_p20, set(), _pending_functions_p20)
    for _cell_p20 in (_function_p20.__closure__ or ()):
        _audit_value_p20(_cell_p20.cell_contents, set(), _pending_functions_p20)

    _code_pending_p20 = [_function_p20.__code__]
    while _code_pending_p20:
        _code_p20 = _code_pending_p20.pop()
        _codes_p20.append(_code_p20)
        _code_pending_p20.extend(
            item for item in _code_p20.co_consts
            if isinstance(item, types.CodeType)
        )
        _names_p20 = {name.lower() for name in _code_p20.co_names}
        assert not (_names_p20 & {
            "inv", "pinv", "lstsq", "getattr", "__dict__",
            "sklearn", "statsmodels",
        })
        assert _function_p20.__name__ not in _code_p20.co_names
        _ops_p20 = {item.opname for item in dis.get_instructions(_code_p20)}
        assert "FOR_ITER" not in _ops_p20
        assert not any(name.startswith("JUMP_BACKWARD") for name in _ops_p20)
        for _name_p20 in _code_p20.co_names:
            if _name_p20 in _function_p20.__globals__:
                _global_p20 = _function_p20.__globals__[_name_p20]
                _audit_value_p20(_global_p20, set(), _pending_functions_p20)
                if isinstance(_global_p20, types.FunctionType):
                    _pending_functions_p20.append(_global_p20)

try:
    _source_p20 = inspect.getsource(ols_full_rank).lower()
except (OSError, TypeError):
    _source_p20 = ""
assert all(token not in _source_p20 for token in (
    "np.linalg.inv(", "np.linalg.pinv(", "np.linalg.lstsq(",
    "getattr(", "__dict__", "sklearn", "statsmodels",
))

_X1_p20 = np.array([
    [1.0, -2.0, 0.0],
    [1.0, 0.0, 1.0],
    [1.0, 1.0, -1.0],
    [1.0, 3.0, 2.0],
    [1.0, 4.0, 0.0],
])
_y1_p20 = np.array([-0.2, 0.8, 2.2, 6.4, 6.3])
_X2_p20 = np.array([
    [1.0, -3.0],
    [1.0, -1.0],
    [1.0, 2.0],
    [1.0, 4.0],
    [1.0, 6.0],
    [1.0, 9.0],
])
_y2_p20 = np.array([-4.0, -0.5, 4.2, 7.1, 10.5, 14.8])
_X3_p20 = np.array([
    [1.0, 1.0],
    [1.0, 1.0 + 3e-7],
    [1.0, 1.0 - 3e-7],
    [1.0, 1.0 + 6e-7],
])
_y3_p20 = np.arange(4.0)
_accepted_p20 = (
    (_X1_p20, _y1_p20, True),
    (_X2_p20, _y2_p20, True),
    (_X2_p20 * 1e8, _y2_p20 * 1e8, True),
    (_X3_p20, _y3_p20, False),
)


def _invoke_p20(X, y, solve_replacement):
    forbidden_calls = []

    def forbid(name):
        def blocked(*args, **kwargs):
            forbidden_calls.append(name)
            raise AssertionError(f"forbidden route called: {name}")
        return blocked

    np.linalg.solve = solve_replacement
    np.linalg.inv = forbid("inv")
    np.linalg.pinv = forbid("pinv")
    np.linalg.lstsq = forbid("lstsq")
    try:
        result = ols_full_rank(X, y)
    finally:
        np.linalg.solve = _ORIGINAL_SOLVE_P20
        np.linalg.inv = _ORIGINAL_INV_P20
        np.linalg.pinv = _ORIGINAL_PINV_P20
        np.linalg.lstsq = _ORIGINAL_LSTSQ_P20
    assert forbidden_calls == []
    return result


for _X_p20, _y_p20, _compare_lstsq_p20 in _accepted_p20:
    _G_p20 = _X_p20.T @ _X_p20
    _c_p20 = _X_p20.T @ _y_p20
    _expected_p20 = _ORIGINAL_SOLVE_P20(_G_p20, _c_p20)
    if _compare_lstsq_p20:
        _lstsq_p20 = _ORIGINAL_LSTSQ_P20(_X_p20, _y_p20, rcond=None)[0]
        assert np.allclose(
            _expected_p20, _lstsq_p20, atol=ATOL, rtol=RTOL,
        )
    _solve_calls_p20 = []

    def counted_solve(a, b):
        _solve_calls_p20.append(
            (np.array(a, copy=True), np.array(b, copy=True))
        )
        return _ORIGINAL_SOLVE_P20(a, b)

    _X_before_p20 = _X_p20.copy()
    _y_before_p20 = _y_p20.copy()
    _beta_p20 = _invoke_p20(_X_p20, _y_p20, counted_solve)
    assert len(_solve_calls_p20) == 1
    assert np.array_equal(_solve_calls_p20[0][0], _G_p20)
    assert np.array_equal(_solve_calls_p20[0][1], _c_p20)
    assert isinstance(_beta_p20, np.ndarray)
    assert _beta_p20.shape == (_X_p20.shape[1],)
    assert np.issubdtype(_beta_p20.dtype, np.floating)
    assert np.isfinite(_beta_p20).all()
    assert np.array_equal(_X_p20, _X_before_p20)
    assert np.array_equal(_y_p20, _y_before_p20)
    assert np.allclose(
        _beta_p20, _expected_p20, atol=ATOL, rtol=RTOL,
    )
    _resid_p20 = _X_p20 @ _beta_p20 - _y_p20
    _expected_resid_p20 = _X_p20 @ _expected_p20 - _y_p20
    assert np.allclose(
        _resid_p20, _expected_resid_p20, atol=ATOL, rtol=RTOL,
    )
    _normal_gap_p20 = np.linalg.norm(
        _G_p20 @ _beta_p20 - _c_p20, ord=np.inf,
    )
    _normal_scale_p20 = (
        np.linalg.norm(_G_p20, ord=np.inf)
        * np.linalg.norm(_beta_p20, ord=np.inf)
        + np.linalg.norm(_c_p20, ord=np.inf)
    )
    _raw_cond_p20 = np.linalg.cond(_G_p20)
    _effective_cond_p20 = min(
        _raw_cond_p20 if np.isfinite(_raw_cond_p20) else 1 / _FLOAT_EPS_P20,
        1 / _FLOAT_EPS_P20,
    )
    _normal_bound_p20 = (
        ATOL
        + _BACKWARD_SAFETY_P20
        * _FLOAT_EPS_P20
        * max(1.0, _effective_cond_p20)
        * max(1.0, _normal_scale_p20)
    )
    assert np.isfinite(_normal_bound_p20)
    assert _normal_gap_p20 <= _normal_bound_p20
    _orth_gap_p20 = np.linalg.norm(
        _X_p20.T @ _resid_p20, ord=np.inf,
    )
    _orth_scale_p20 = (
        np.linalg.norm(_X_p20.T, ord=np.inf)
        * np.linalg.norm(_resid_p20, ord=np.inf)
    )
    _orth_bound_p20 = (
        ATOL
        + _BACKWARD_SAFETY_P20
        * _FLOAT_EPS_P20
        * max(1.0, _effective_cond_p20)
        * max(1.0, _orth_scale_p20)
    )
    assert np.isfinite(_orth_bound_p20)
    assert _orth_gap_p20 <= _orth_bound_p20

_G_flow_p20 = _X2_p20.T @ _X2_p20
_c_flow_p20 = _X2_p20.T @ _y2_p20
_sentinel_p20 = np.array([123.25, -77.5])
_flow_calls_p20 = []


def flow_solve_p20(a, b):
    _flow_calls_p20.append((np.array(a, copy=True), np.array(b, copy=True)))
    return _sentinel_p20.copy()


_flow_beta_p20 = _invoke_p20(_X2_p20, _y2_p20, flow_solve_p20)
assert len(_flow_calls_p20) == 1
assert np.array_equal(_flow_calls_p20[0][0], _G_flow_p20)
assert np.array_equal(_flow_calls_p20[0][1], _c_flow_p20)
assert np.array_equal(_flow_beta_p20, _sentinel_p20)

_pre_rejected_p20 = (
    (np.ones(3), np.ones(3)),
    (np.ones((3, 2)), np.ones((3, 1))),
    (np.ones((3, 2)), np.ones(2)),
    (np.ones((2, 3)), np.ones(2)),
    (np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]]), np.ones(3)),
    (np.array([[1.0, np.nan], [1.0, 2.0]]), np.ones(2)),
)
for _X_bad_p20, _y_bad_p20 in _pre_rejected_p20:
    _solve_calls_p20 = []

    def unexpected_solve_p20(a, b):
        _solve_calls_p20.append((a, b))
        return _ORIGINAL_SOLVE_P20(a, b)

    try:
        _invoke_p20(_X_bad_p20, _y_bad_p20, unexpected_solve_p20)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input must raise ValueError")
    assert _solve_calls_p20 == []

_X_gram_singular_p20 = np.array([
    [1.0, 1.0],
    [1.0, 1.0 + 1e-9],
    [1.0, 1.0 - 1e-9],
    [1.0, 1.0 + 2e-9],
])
_y_gram_singular_p20 = np.arange(4.0)
assert np.linalg.matrix_rank(_X_gram_singular_p20) == 2
_singular_calls_p20 = []


def singular_solve_p20(a, b):
    _singular_calls_p20.append(
        (np.array(a, copy=True), np.array(b, copy=True))
    )
    return _ORIGINAL_SOLVE_P20(a, b)


try:
    _invoke_p20(
        _X_gram_singular_p20, _y_gram_singular_p20, singular_solve_p20,
    )
except ValueError:
    pass
else:
    raise AssertionError("LinAlgError must be converted to ValueError")
assert len(_singular_calls_p20) == 1
assert np.array_equal(
    _singular_calls_p20[0][0],
    _X_gram_singular_p20.T @ _X_gram_singular_p20,
)
assert np.array_equal(
    _singular_calls_p20[0][1],
    _X_gram_singular_p20.T @ _y_gram_singular_p20,
)